## Data Set Cleaning 

In [11]:
import pandas as pd

# Load the dataset
file_path = "HydroSense_Data - Sheet5.csv"

cols = [
    "timestamp", "flow1", "flow2", "flow3", "pressure",
    "diff12", "diff23", "ratio12", "ratio23",
    "flag_old", "zone_old", "severity_old", "status_old", "extra"
]

df = pd.read_csv(file_path, header=None, names=cols)

In [14]:

if "extra" in df.columns and df["extra"].isna().all():
    df = df.drop(columns=["extra"])

In [15]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

C:\Users\shubh\AppData\Local\Temp\ipykernel_49016\176417918.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


In [17]:
numeric_cols = [
    "flow1", "flow2", "flow3", "pressure",
    "diff12", "diff23", "ratio12", "ratio23", "flag_old"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [18]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Shape: (1135, 14)

Missing values:
timestamp          1
flow1              1
flow2              1
flow3              1
pressure           1
diff12             1
diff23             1
ratio12            1
ratio23            1
flag_old           1
zone_old           0
severity_old       0
status_old         0
extra           1134
dtype: int64


In [19]:
print("\nData types:")
print(df.dtypes)


Data types:
timestamp       datetime64[ns]
flow1                  float64
flow2                  float64
flow3                  float64
pressure               float64
diff12                 float64
diff23                 float64
ratio12                float64
ratio23                float64
flag_old               float64
zone_old                object
severity_old            object
status_old              object
extra                   object
dtype: object


In [20]:
df.to_csv("hydrosense_phase0_cleaned.csv", index=False)
print("\nPhase 0 complete. Saved as hydrosense_phase0_cleaned.csv")


Phase 0 complete. Saved as hydrosense_phase0_cleaned.csv


## Calibaration Check and Relabeling 

In [22]:
df = pd.read_csv("hydrosense_phase0_cleaned.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

In [23]:
def get_operating_state(row):
    max_flow = max(row["flow1"], row["flow2"], row["flow3"])
    if max_flow < 0.1:
        return "NO_FLOW"
    elif max_flow < 1.0:
        return "LOW_FLOW"
    else:
        return "ACTIVE_FLOW"

In [24]:
df["operating_state"] = df.apply(get_operating_state, axis=1)

In [31]:
active_df = df[df["operating_state"] == "ACTIVE_FLOW"].copy()

In [32]:
# From active flow, keep old NORMAL rows for baseline study
baseline_df = active_df[active_df["zone_old"] == "NORMAL"].copy()

In [33]:
print("Total rows:", len(df))
print("Active rows:", len(active_df))
print("Baseline NORMAL active rows:", len(baseline_df))

Total rows: 1135
Active rows: 918
Baseline NORMAL active rows: 104


In [34]:
 # 6. Mean flow of each sensor in baseline region
print("\nBaseline means:")
print(baseline_df[["flow1", "flow2", "flow3"]].mean())

print("\nBaseline std dev:")
print(baseline_df[["flow1", "flow2", "flow3"]].std())


Baseline means:
flow1    1.371346
flow2    1.794423
flow3    1.611538
dtype: float64

Baseline std dev:
flow1    0.546761
flow2    0.566340
flow3    0.498699
dtype: float64


In [35]:
baseline_df["flow2_minus_flow1"] = baseline_df["flow2"] - baseline_df["flow1"]
baseline_df["flow2_minus_flow3"] = baseline_df["flow2"] - baseline_df["flow3"]

print("\nAverage flow2 - flow1:", baseline_df["flow2_minus_flow1"].mean())
print("Average flow2 - flow3:", baseline_df["flow2_minus_flow3"].mean())


Average flow2 - flow1: 0.42307692307692313
Average flow2 - flow3: 0.18288461538461542


In [36]:
# 8. Optional calibration factors using baseline mean
mean_f1 = baseline_df["flow1"].mean()
mean_f2 = baseline_df["flow2"].mean()
mean_f3 = baseline_df["flow3"].mean()

In [37]:
k1 = mean_f1 / mean_f1 if mean_f1 != 0 else 1
k2 = mean_f1 / mean_f2 if mean_f2 != 0 else 1
k3 = mean_f1 / mean_f3 if mean_f3 != 0 else 1

print("\nSuggested calibration factors:")
print("k1 =", k1)
print("k2 =", k2)
print("k3 =", k3)


Suggested calibration factors:
k1 = 1.0
k2 = 0.764226770978459
k3 = 0.8509546539379474


In [40]:
import numpy as np
#9. Create calibrated columns
df["flow1_cal"] = df["flow1"] * k1
df["flow2_cal"] = df["flow2"] * k2
df["flow3_cal"] = df["flow3"] * k3

# 10. Recompute calibrated diffs and ratios
df["diff12_cal"] = abs(df["flow1_cal"] - df["flow2_cal"])
df["diff23_cal"] = abs(df["flow2_cal"] - df["flow3_cal"])

df["ratio12_cal"] = np.where(df["flow2_cal"] != 0, df["flow1_cal"] / df["flow2_cal"], np.nan)
df["ratio23_cal"] = np.where(df["flow3_cal"] != 0, df["flow2_cal"] / df["flow3_cal"], np.nan)

In [42]:
df.to_csv("hydrosense_phase1_prepared.csv", index=False)
print("Saved as hydrosense_phase1_prepared.csv")

Saved as hydrosense_phase1_prepared.csv


In [46]:
df[["flow1_cal", "flow2_cal", "flow3_cal"]].mean()


flow1_cal    1.283157
flow2_cal    1.198151
flow3_cal    1.391536
dtype: float64

In [47]:
df[["diff12_cal", "diff23_cal"]].describe()

,diff12_cal,diff23_cal
count,1134.000000,1134.000000
mean,0.259630,0.294458
std,0.419684,0.400926
min,0.000000,0.000000
25%,0.067559,0.052029
50%,0.120648,0.175762
75%,0.170568,0.296263
max,1.772358,1.752967


## Relabeleing 

In [48]:
import pandas as pd

# Load phase 1 dataset
df = pd.read_csv("hydrosense_phase1_prepared.csv")


In [49]:
def classify(row):
    d12 = row["diff12_cal"]
    d23 = row["diff23_cal"]

    max_diff = max(d12, d23)

    # Severity
    if max_diff >= 0.30:
        severity = "HIGH"
    elif max_diff >= 0.15:
        severity = "MEDIUM"
    else:
        severity = "LOW"

    # Zone
    if severity == "LOW":
        zone = "NORMAL"
    else:
        if d12 > d23:
            zone = "LEAK_1_2"
        else:
            zone = "LEAK_2_3"

    return pd.Series([zone, severity])


In [50]:
# Apply classification
df[["zone_new", "severity_new"]] = df.apply(classify, axis=1)

# Save new dataset
df.to_csv("hydrosense_phase2_labeled.csv", index=False)

print("Phase 2 complete")
print(df[["zone_new", "severity_new"]].value_counts())

Phase 2 complete
zone_new  severity_new
NORMAL    LOW             360
LEAK_2_3  MEDIUM          340
          HIGH            189
LEAK_1_2  MEDIUM          125
          HIGH            121
Name: count, dtype: int64


In [51]:
df.describe()

,flow1,flow2,flow3,pressure,diff12,diff23,ratio12,ratio23,flag_old,flow1_cal,flow2_cal,flow3_cal,diff12_cal,diff23_cal,ratio12_cal,ratio23_cal
count,1134.000000,1134.000000,1134.000000,1134.0,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,907.000000,1008.000000
mean,1.283157,1.567795,1.635265,1.0,0.610071,0.350732,1.175569,0.940298,0.716931,1.283157,1.198151,1.391536,0.259630,0.294458,2.225821,0.934407
std,0.666170,1.041540,0.890918,0.0,0.369762,0.474320,0.475947,0.440088,0.450688,0.666170,0.795973,0.758131,0.419684,0.400926,15.927755,1.203253
min,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.995000,0.290000,1.280000,1.0,0.470000,0.060000,1.000000,0.881772,0.000000,0.995000,0.221626,1.089222,0.067559,0.052029,0.899818,0.841244
50%,1.370000,1.920000,1.850000,1.0,0.600000,0.190000,1.380083,0.988508,1.000000,1.370000,1.467315,1.574266,0.120648,0.175762,0.932503,0.926936
75%,1.837500,2.490000,2.367500,1.0,0.750000,0.380000,1.420011,1.000000,1.000000,1.837500,1.902925,2.014635,0.170568,0.296263,0.956972,1.024141
max,2.170000,2.880000,3.000000,1.0,1.770000,2.060000,2.416667,10.222222,1.000000,2.170000,2.200973,2.552864,1.772358,1.752967,232.915159,22.452042


In [52]:
df["zone_new"].value_counts()

zone_new
LEAK_2_3    529
NORMAL      360
LEAK_1_2    246
Name: count, dtype: int64